# IEEE-CIS Fraud Detection — AdaBoost

Sections: **Cleaning → Feature Engineering → Feature Selection → Training**

In [1]:
!pip install dagshub mlflow scikit-learn pandas numpy -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/

In [2]:
import os, gc, warnings
import numpy as np
import pandas as pd
import mlflow, mlflow.sklearn
warnings.filterwarnings('ignore')

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from mlflow.models.signature import infer_signature
from scipy.stats import randint, uniform

def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min(); c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['MLFLOW_TRACKING_USERNAME'] = 'dgrig23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = secrets.get_secret('DAGSHUB_TOKEN')

REPO = 'dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning'
mlflow.set_tracking_uri(f'https://dagshub.com/{REPO}.mlflow')
mlflow.set_experiment('AdaBoost_Training')
EXP_PREFIX = 'AdaBoost'
BASE = '/kaggle/input/competitions/ieee-fraud-detection/'
print('Ready.')

Ready.


## 1. Cleaning

In [3]:
train_trx = reduce_mem_usage(pd.read_csv(BASE + 'train_transaction.csv'))
train_idn = reduce_mem_usage(pd.read_csv(BASE + 'train_identity.csv'))

train_idn.columns = train_idn.columns.str.replace('-', '_')
train = train_trx.merge(train_idn, on='TransactionID', how='left')
del train_trx, train_idn; gc.collect()
print(f'Train shape: {train.shape}')

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Cleaning'):
    HIGH_MISS = 0.9
    miss = train.isnull().mean()
    high_miss_cols = miss[miss > HIGH_MISS].index.tolist()
    
    train.drop(columns=high_miss_cols + ['TransactionID'], inplace=True, errors='ignore')
    
    y = train.pop('isFraud').copy()
    fraud_rate = y.mean()
    class_weight_pos = round((1 - fraud_rate) / fraud_rate, 2)
    
    sample_weights = np.where(y == 1, class_weight_pos, 1.0)
    
    mlflow.log_params({'high_miss_threshold': HIGH_MISS, 'cols_dropped': len(high_miss_cols), 'class_weight_pos': class_weight_pos})
    mlflow.log_metrics({'fraud_rate': round(float(fraud_rate), 4), 'cols_after': train.shape[1]})
    
    print(f'Fraud rate: {fraud_rate:.4f} | class_weight_pos: {class_weight_pos}')

Train shape: (590540, 434)
Fraud rate: 0.0350 | class_weight_pos: 27.58
🏃 View run AdaBoost_Cleaning at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1/runs/470ed3f909fa469185cd102a49cec7bc
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1


## 2. Feature Engineering

In [4]:
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Engineering'):
    cols_before = train.shape[1]
    
    if 'TransactionDT' in train.columns:
        train['hour']      = ((train['TransactionDT'] / 3600)        % 24).astype(np.float32)
        train['dayofweek'] = ((train['TransactionDT'] / (3600*24))   %  7).astype(np.float32)
        train['week']      = ((train['TransactionDT'] / (3600*24*7)) % 52).astype(np.float32)
        train.drop(columns=['TransactionDT'], inplace=True)
    
    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in train.columns:
            s = train[col].str.split('.', expand=False)
            train[col+'_suffix'] = s.str[-1].fillna('unknown')
            train[col+'_domain'] = s.str[0].fillna('unknown')
            
    if 'P_emaildomain' in train.columns and 'R_emaildomain' in train.columns:
        train['email_match'] = (train['P_emaildomain'] == train['R_emaildomain']).astype(np.int8)
        
    if 'TransactionAmt' in train.columns:
        train['TransactionAmt_log']   = np.log1p(train['TransactionAmt']).astype(np.float32)
        train['TransactionAmt_cents'] = (train['TransactionAmt'] % 1).astype(np.float32)
        train['amt_is_round']         = (train['TransactionAmt'] % 1    == 0).astype(np.int8)
        train['amt_is_round_100']     = (train['TransactionAmt'] % 100 == 0).astype(np.int8)
        
    for g in ['card1', 'card4', 'addr1']:
        if g in train.columns:
            train[f'{g}_amt_mean'] = train.groupby(g)['TransactionAmt'].transform('mean').astype(np.float32)
            train[f'{g}_amt_std']  = train.groupby(g)['TransactionAmt'].transform('std').fillna(0).astype(np.float32)
            train[f'{g}_amt_zscore'] = ((train['TransactionAmt'] - train[f'{g}_amt_mean']) / 
                                        train[f'{g}_amt_std'].replace(0,1)).clip(-5,5).astype(np.float32)

    uid_parts = [c for c in ['card1','card2','addr1','P_emaildomain'] if c in train.columns]
    train['user_id'] = train[uid_parts[0]].astype(str)
    for c in uid_parts[1:]:
        train['user_id'] += "_" + train[c].astype(str)
    
    train['uid_count'] = train.groupby('user_id')['TransactionAmt'].transform('count').astype(np.int32)
    train['uid_mean']  = train.groupby('user_id')['TransactionAmt'].transform('mean').astype(np.float32)
    train['uid_std']   = train.groupby('user_id')['TransactionAmt'].transform('std').fillna(0).astype(np.float32)
    
    train['user_count_log'] = np.log1p(train['uid_count']).astype(np.float32)
    train.drop(columns=['user_id'], inplace=True)

    mlflow.log_params({'time':'hour,dow,week','email':'suffix,domain,match','amount':'log,cents,round','aggs':'card1,card4,addr1,user_id'})
    mlflow.log_metrics({'new_features': train.shape[1]-cols_before, 'total_features': train.shape[1]})
    print(f'Features: {cols_before} → {train.shape[1]}')

Features: 420 → 444
🏃 View run AdaBoost_Feature_Engineering at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1/runs/9494c1f8a73b455a8cf371ac3bc0b9e0
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1


## 3. Feature Selection

In [5]:
cat_cols = train.select_dtypes(include='object').columns.tolist()
le_store = {}

for col in cat_cols:
    train[col] = train[col].fillna('unknown').astype(str)
    le = LabelEncoder()
    le.fit(list(train[col].unique()) + ['unknown'])
    le_store[col] = le
    train[col] = le.transform(train[col]).astype(np.int32)

num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
medians = train[num_cols].median()
train[num_cols] = train[num_cols].fillna(medians).astype(np.float32)

print(f'Missing: {train.isnull().sum().sum()}')

Missing: 0


In [6]:
X_tr, X_va, y_tr, y_va = train_test_split(train, y, test_size=0.2, stratify=y, random_state=42)
sw_tr = np.where(y_tr == 1, class_weight_pos, 1.0)

def quick_eval(features):

    m = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=2),
        n_estimators=30, 
        learning_rate=1.0, 
        random_state=42
    )
    m.fit(X_tr[features], y_tr, sample_weight=sw_tr)

    return roc_auc_score(y_va, m.predict_proba(X_va[features])[:,1])

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Selection'):
    all_features = train.columns.tolist()
    auc_all = quick_eval(all_features)
    print(f'Strategy A – All ({len(all_features)}): AUC = {auc_all:.5f}')

    vt = VarianceThreshold(threshold=0.01)
    vt.fit(train)
    vt_features = train.columns[vt.get_support()].tolist()
    auc_vt = quick_eval(vt_features)
    print(f'Strategy B – VarianceThreshold ({len(vt_features)}): AUC = {auc_vt:.5f}')

    corr = train.corrwith(y).abs()
    corr_features = corr[corr >= 0.01].index.tolist()
    auc_corr = quick_eval(corr_features)
    print(f'Strategy C – Correlation filter ({len(corr_features)}): AUC = {auc_corr:.5f}')

    best_strategy, best_auc, final_features = max(
        [('all', auc_all, all_features), 
         ('variance_threshold', auc_vt, vt_features), 
         ('correlation_filter', auc_corr, corr_features)],
        key=lambda x: x[1]
    )
    
    mlflow.log_params({
        'strategy_A': 'all',
        'strategy_B': 'variance_threshold_0.01',
        'strategy_C': 'correlation_0.01',
        'selected': best_strategy,
        'n_final': len(final_features)
    })
    mlflow.log_metrics({
        'auc_all': round(auc_all, 5),
        'auc_vt': round(auc_vt, 5),
        'auc_corr': round(auc_corr, 5),
        'best_auc': round(best_auc, 5)
    })
    print(f'→ Best: {best_strategy} | AUC: {best_auc:.5f} | Features: {len(final_features)}')

Strategy A – All (444): AUC = 0.86663
Strategy B – VarianceThreshold (419): AUC = 0.86663
Strategy C – Correlation filter (325): AUC = 0.85966
→ Best: all | AUC: 0.86663 | Features: 444
🏃 View run AdaBoost_Feature_Selection at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1/runs/74a7afd3812246d1bae81281f0c0f1fc
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1


## 4. Training — AdaBoost

In [7]:
X = train[final_features].copy()
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
sw_tr = np.where(y_tr == 1, class_weight_pos, 1.0)

# (a) Underfitting
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Underfit_Config'):
    m = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=10, 
        learning_rate=0.1, 
        random_state=42
    )
    m.fit(X_tr, y_tr, sample_weight=sw_tr)
    
    tr_auc = roc_auc_score(y_tr, m.predict_proba(X_tr)[:,1])
    va_auc = roc_auc_score(y_va, m.predict_proba(X_va)[:,1])
    
    mlflow.log_params({'n_estimators':10,'learning_rate':0.1,'base_depth':1,'note':'stump_underfit'})
    mlflow.log_metrics({'train_auc':round(tr_auc,5),'val_auc':round(va_auc,5),'overfit_gap':round(tr_auc-va_auc,5)})
    print(f'Underfit — Train: {tr_auc:.5f} | Val: {va_auc:.5f}')

# (b) Overfitting
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Overfit_Config'):
    n_samples = len(X_tr)
    indices = np.arange(n_samples)
    tiny_idx = np.random.choice(indices, size=int(0.1 * n_samples), replace=False)

    X_tr_tiny = X_tr.iloc[tiny_idx]
    y_tr_tiny = y_tr.iloc[tiny_idx]
    sw_tr_tiny = sw_tr[tiny_idx]

    m = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=5), 
        n_estimators=200, 
        learning_rate=1.5, 
        random_state=42
    )
    
    m.fit(X_tr_tiny, y_tr_tiny, sample_weight=sw_tr_tiny)
    
    tr_auc = roc_auc_score(y_tr_tiny, m.predict_proba(X_tr_tiny)[:,1])
    va_auc = roc_auc_score(y_va, m.predict_proba(X_va)[:,1])
    
    mlflow.log_params({
        'n_estimators': 200, 
        'learning_rate': 1.5, 
        'base_depth': 5, 
        'actual_train_size': '10% of 80%'
    })
    
    mlflow.log_metrics({
        'train_auc': round(tr_auc, 5),
        'val_auc': round(va_auc, 5),
        'overfit_gap': round(tr_auc - va_auc, 5)
    })
    
    print(f'Overfit — Train: {tr_auc:.5f} | Val: {va_auc:.5f} | Gap: {tr_auc-va_auc:.5f}')

Underfit — Train: 0.75053 | Val: 0.75482
🏃 View run AdaBoost_Underfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1/runs/9166234521764227871d6e9e9d9ccd90
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1
Overfit — Train: 1.00000 | Val: 0.87941 | Gap: 0.12059
🏃 View run AdaBoost_Overfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1/runs/1ec66a721c364b0b9cd3d0d78b0ce272
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1


In [8]:
# (c) RandomizedSearch 
with mlflow.start_run(run_name=f'{EXP_PREFIX}_RandomizedSearch') as run_rs:

    X_rs = X.sample(frac=0.2, random_state=42)
    y_rs = y.loc[X_rs.index]
    sw_rs = sample_weights[X_rs.index]
    
    param_dist = {
        'n_estimators':                [50, 100, 200],   
        'learning_rate':               [0.01, 0.1, 0.5],
        'estimator__max_depth':        [1],
        'estimator__min_samples_leaf': randint(1, 30),
    }
    
    base_clf = AdaBoostClassifier(estimator=DecisionTreeClassifier(), random_state=42)
    skf3 = StratifiedKFold(3, shuffle=True, random_state=42)
    
    rs = RandomizedSearchCV(
        base_clf, param_dist, n_iter=10, cv=skf3,
        scoring='roc_auc', n_jobs=-1, return_train_score=True,
        random_state=42, verbose=1
    )
    
    rs.fit(X_rs, y_rs, sample_weight=sw_rs)

    for i, params in enumerate(rs.cv_results_['params']):
        label = f"n{params['n_estimators']}_lr{round(params['learning_rate'],2)}_d{params['estimator__max_depth']}"
        with mlflow.start_run(run_name=f'AdaBoost_RS_{label}', nested=True):
            mlflow.log_params(params)
            mlflow.log_metrics({
                'cv_auc_mean':    round(float(rs.cv_results_['mean_test_score'][i]),  5),
                'cv_auc_std':     round(float(rs.cv_results_['std_test_score'][i]),   5),
                'train_auc_mean': round(float(rs.cv_results_['mean_train_score'][i]), 5),
                'overfit_gap':    round(float(rs.cv_results_['mean_train_score'][i]-rs.cv_results_['mean_test_score'][i]), 5),
            })

    best_params = rs.best_params_
    mlflow.log_params({'best_'+k: v for k,v in best_params.items()})
    mlflow.log_metric('best_cv_auc', round(rs.best_score_, 5))
    
    print(f'Best CV AUC: {rs.best_score_:.5f} | Params: {best_params}')

Fitting 3 folds for each of 10 candidates, totalling 30 fits
🏃 View run AdaBoost_RS_n200_lr0.01_d1 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1/runs/48ca292ed4254632903d3037b0593e02
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1
🏃 View run AdaBoost_RS_n50_lr0.01_d1 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1/runs/747ab4b5f937467485cbb77c1878e25d
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1
🏃 View run AdaBoost_RS_n200_lr0.1_d1 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1/runs/299c2a486ff04fdcb3a078e0a61ae0c4
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1
🏃 View run AdaBoost_RS_n200_lr0.5_d1 at: https://dagshub.com/dgri

In [9]:
# (d) Final Pipeline
class FraudPreprocessorAdaBoost(BaseEstimator, TransformerMixin):
    def __init__(self, miss_thresh=0.9):
        self.miss_thresh=miss_thresh; 
        self.high_miss_cols_=[]; 
        self.le_store_={}; 
        self.medians_=None; 
        self.features_=None
        
    def fit(self, X, y=None):
        X=X.copy(); X.columns=X.columns.str.replace('-','_')
        for c in ['TransactionID','isFraud']:
            if c in X.columns: X.drop(columns=[c], inplace=True)
            
        self.high_miss_cols_=X.columns[X.isnull().mean()>self.miss_thresh].tolist()
        X.drop(columns=self.high_miss_cols_, inplace=True, errors='ignore')
        X=self._engineer(X)
        
        for col in X.select_dtypes(include='object').columns:
            le=LabelEncoder(); vals=X[col].fillna('unknown').astype(str)
            le.fit(list(vals.unique())+['unknown']); 
            self.le_store_[col]=le; 
            X[col]=le.transform(vals).astype(np.int32)
            
        num=X.select_dtypes(include=[np.number]).columns.tolist()
        self.medians_=X[num].median()
        X[num]=X[num].fillna(self.medians_).astype(np.float32)
        
        self.features_=X.columns.tolist(); 
        return self
        
    def transform(self, X):
        X=X.copy(); X.columns=X.columns.str.replace('-','_')
        for c in ['TransactionID','isFraud']:
            if c in X.columns: X.drop(columns=[c], inplace=True)
            
        X.drop(columns=self.high_miss_cols_, inplace=True, errors='ignore')
        X=self._engineer(X)
        
        for col in X.select_dtypes(include='object').columns:
            if col in self.le_store_:
                known=set(self.le_store_[col].classes_)
                X[col]=X[col].fillna('unknown').astype(str)
                # Fastened: Replaced .apply(lambda) with vectorized .isin()
                X.loc[~X[col].isin(known), col] = 'unknown'
                X[col]=self.le_store_[col].transform(X[col]).astype(np.int32)
                
        num=X.select_dtypes(include=[np.number]).columns.tolist()
        X[num]=X[num].fillna(self.medians_).astype(np.float32)
        
        for f in self.features_:
            if f not in X.columns: X[f]=0
        return X[self.features_]
        
    def _engineer(self, df):
        if 'TransactionDT' in df.columns:
            df['hour']=((df['TransactionDT']/3600)%24).astype(np.float32)
            df['dayofweek']=((df['TransactionDT']/(3600*24))%7).astype(np.float32)
            df.drop(columns=['TransactionDT'], inplace=True)
            
        for col in ['P_emaildomain','R_emaildomain']:
            if col in df.columns:
                df[col+'_suffix']=df[col].str.split('.', expand=False).str[-1].fillna('unknown')
                
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match']=(df['P_emaildomain']==df['R_emaildomain']).astype(np.int8)
            
        if 'TransactionAmt' in df.columns:
            df['TransactionAmt_log']=np.log1p(df['TransactionAmt']).astype(np.float32)
            df['amt_is_round']=(df['TransactionAmt']%1==0).astype(np.int8)
        return df

print('Reloading raw data...')
raw_trx=reduce_mem_usage(pd.read_csv(BASE+'train_transaction.csv')) 
raw_idn=reduce_mem_usage(pd.read_csv(BASE+'train_identity.csv'))

raw_train=raw_trx.merge(raw_idn, on='TransactionID', how='left'); 
y_raw=raw_train['isFraud'].copy()
sw_raw=np.where(y_raw==1, class_weight_pos, 1.0); 
del raw_trx, raw_idn; gc.collect()

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Final_Model'):
    final_base = DecisionTreeClassifier(max_depth=best_params['estimator__max_depth'], min_samples_leaf=best_params['estimator__min_samples_leaf'])
    final_clf  = AdaBoostClassifier(estimator=final_base, n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], random_state=42)
    
    final_pipeline = Pipeline([('preprocessor', FraudPreprocessorAdaBoost()), ('classifier', final_clf)])
    final_pipeline.fit(raw_train, y_raw, classifier__sample_weight=sw_raw)
    
    mlflow.log_params({**best_params, 'miss_thresh': 0.9})
    mlflow.log_metric('final_cv_auc', round(rs.best_score_, 5))
    
    sig=infer_signature(raw_train.head(5), final_pipeline.predict_proba(raw_train.head(5))[:,1])
    mlflow.sklearn.log_model(final_pipeline, artifact_path='adaboost_pipeline', signature=sig, registered_model_name='AdaBoost_FraudDetection')
    
    print(f'Registered. CV AUC: {rs.best_score_:.5f}')

Reloading raw data...


2026/05/07 09:06:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 09:06:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'AdaBoost_FraudDetection' already exists. Creating a new version of this model...
2026/05/07 09:07:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AdaBoost_FraudDetection, version 2
Created version '2' of model 'AdaBoost_FraudDetection'.


Registered. CV AUC: 0.86451
🏃 View run AdaBoost_Final_Model at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1/runs/287431562ca24a27815b2e5cd9e2fc0f
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/1
